# Law RAG Evaluation - Generated Dataset

Notebook n?y ch?y evaluation tr?n b? `evaluation/law_rag_eval_dataset.json` g?m c?c c?u h?i ??i th??ng v? expected source th?t l?y t? corpus.

M?c ti?u:
- ch?y evaluator end-to-end;
- l?u JSON report;
- t?m t?t metric ch?nh;
- xem nh?m/case y?u ?? c?i thi?n retrieval.

In [ ]:
from __future__ import annotations

import json
import statistics
import subprocess
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "evaluation" else Path.cwd().resolve()
DATASET_PATH = ROOT_DIR / "evaluation" / "law_rag_eval_dataset.json"
RUNS_DIR = ROOT_DIR / "output" / "eval" / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("DATASET:", DATASET_PATH)
print("RUNS_DIR:", RUNS_DIR)


## 1. Dataset Overview

In [ ]:
dataset = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
print("case_count:", len(dataset))
print("categories:")
for category, count in Counter(item["category"] for item in dataset).most_common():
    print(f"- {category}: {count}")

print("\nSample cases:")
for item in dataset[:5]:
    expected = item["reference_outputs"]["expected_sources"][0]
    print(f"- {item['id']}: {item['inputs']['question']} -> {expected}")


## 2. Run Evaluation

Ch?y full 56 case b?ng `hybrid --no-query-rewrite`. L? do t?t query rewrite ? l?n ch?y n?y: gi?m s? l?n g?i LLM/API nh?ng v?n ??nh gi? ???c retrieval hybrid + answer generation tr?n to?n b? dataset. N?u mu?n benchmark c?u h?nh production h?n, ??i `NO_QUERY_REWRITE = False`.

In [ ]:
RUN_BASELINE_EVAL = False
NO_QUERY_REWRITE = True
LIMIT = None  # ??t s? nh?, v? d? 10, n?u mu?n ch?y nhanh

if RUN_BASELINE_EVAL:
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    output_path = RUNS_DIR / f"generated-dataset-hybrid-{timestamp}.json"
    cmd = [
        sys.executable,
        str(ROOT_DIR / "evaluation" / "evaluate_law_rag.py"),
        "--dataset", str(DATASET_PATH),
        "--retrieval-mode", "hybrid",
        "--top-k", "5",
        "--output", str(output_path),
    ]
    if NO_QUERY_REWRITE:
        cmd.append("--no-query-rewrite")
    if LIMIT is not None:
        cmd.extend(["--limit", str(LIMIT)])

    print("Running:", " ".join(cmd))
    started = time.time()
    completed = subprocess.run(cmd, cwd=ROOT_DIR, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:\n", completed.stderr)
    print("elapsed_seconds:", round(time.time() - started, 2))
    if completed.returncode != 0:
        raise RuntimeError(f"Evaluation failed with exit code {completed.returncode}")
else:
    output_path = ROOT_DIR / "output" / "eval" / "runs" / "generated-dataset-hybrid-20260613-154936.json"
    timestamp = "20260613-154936"
    print("Skipping baseline evaluation; loading existing report:", output_path)

print("REPORT_PATH", output_path)


## 3. Load Report

In [ ]:
report = json.loads(output_path.read_text(encoding="utf-8"))
summary = report["summary"]
results = report["results"]
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 4. Category Metrics

In [ ]:
def numeric_score(item, key):
    value = item.get("scores", {}).get(key)
    return value if isinstance(value, (int, float)) else None

category_rows = []
for category in sorted({item.get("category") for item in results}):
    group = [item for item in results if item.get("category") == category]
    row = {"category": category, "count": len(group)}
    for metric in ["overall", "recall_at_5", "mrr", "citation_accuracy", "groundedness", "answer_correctness"]:
        values = [numeric_score(item, metric) for item in group]
        values = [v for v in values if v is not None]
        row[metric] = round(statistics.mean(values), 3) if values else None
    category_rows.append(row)

for row in category_rows:
    print(row)


## 5. Lowest Scoring Cases

In [ ]:
lowest = sorted(results, key=lambda item: numeric_score(item, "overall") or 0)[:12]
for item in lowest:
    s = item.get("scores", {})
    print("=" * 100)
    print(item.get("case_id"), "|", item.get("category"), "| overall", s.get("overall"), "recall@5", s.get("recall_at_5"), "citation", s.get("citation_accuracy"))
    print("Q:", item.get("question"))
    print("Expected:", (next((d for d in dataset if d["id"] == item.get("case_id")), {}).get("reference_outputs") or {}).get("expected_sources"))
    print("Top sources:")
    for source in item.get("sources", [])[:3]:
        print(" -", source.get("document_title"), "?i?u", source.get("article_number"), "score", source.get("rerank_score"))


## 6. Save Simple Charts

In [ ]:
import matplotlib.pyplot as plt

chart_dir = ROOT_DIR / "output" / "eval" / "charts"
chart_dir.mkdir(parents=True, exist_ok=True)

metrics = summary.get("scores", {})
selected = ["overall", "recall_at_5", "mrr", "citation_accuracy", "groundedness", "answer_correctness"]
values = [metrics.get(metric, 0) for metric in selected]
plt.figure(figsize=(10, 4))
plt.bar(selected, values)
plt.xticks(rotation=30, ha="right")
plt.title("Law RAG Evaluation Summary")
plt.ylim(0, 5)
summary_chart = chart_dir / f"summary-{timestamp}.png"
plt.tight_layout()
plt.savefig(summary_chart, dpi=160)
plt.show()
print("saved", summary_chart)

labels = [row["category"] for row in category_rows]
overalls = [row.get("overall") or 0 for row in category_rows]
plt.figure(figsize=(11, 4))
plt.bar(labels, overalls)
plt.xticks(rotation=35, ha="right")
plt.title("Overall by Category")
plt.ylim(0, 5)
category_chart = chart_dir / f"category-overall-{timestamp}.png"
plt.tight_layout()
plt.savefig(category_chart, dpi=160)
plt.show()
print("saved", category_chart)


## 7. Interview-Ready Takeaway

In [ ]:
print("Case count:", summary.get("case_count"))
print("Overall:", summary.get("scores", {}).get("overall"))
print("Recall@5:", summary.get("scores", {}).get("recall_at_5"))
print("MRR:", summary.get("scores", {}).get("mrr"))
print("Citation accuracy:", summary.get("scores", {}).get("citation_accuracy"))
print("Groundedness:", summary.get("scores", {}).get("groundedness"))
print("Latency avg ms:", summary.get("latency_ms_avg"))
print("Report:", output_path)


## 8. Query Rewrite Before/After

Ph?n n?y so s?nh c?u h?nh **kh?ng query rewrite** v?i c?u h?nh **c? query rewrite**.

L?u ? quan tr?ng: l?n ch?y query rewrite full 56 case b? d?ng th?c t? do quota OpenAI `429 insufficient_quota` sau 5 case th?nh c?ng. V? v?y b?ng d??i ??y c? hai l?p ??c:

- Full no-rewrite: 56/56 case, d?ng l?m baseline ?n ??nh.
- Rewrite successful subset: ch? 5 case ?? ch?y th?nh c?ng, d?ng ?? xem xu h??ng ban ??u, ch?a ?? k?t lu?n to?n dataset.

??y c?ng l? m?t trade-off th?t c?a query rewrite: ch?t l??ng retrieval c? th? t?ng, nh?ng t?ng s? l?n g?i LLM/API n?n d? t?ng latency v? chi ph?/quota.

In [ ]:
from pathlib import Path
import json, statistics

NO_REWRITE_REPORT = ROOT_DIR / "output" / "eval" / "runs" / "generated-dataset-hybrid-20260613-154936.json"
REWRITE_REPORT = ROOT_DIR / "output" / "eval" / "runs" / "generated-dataset-hybrid-query-rewrite-20260613-rerun.json"

no_rewrite = json.loads(NO_REWRITE_REPORT.read_text(encoding="utf-8"))
rewrite = json.loads(REWRITE_REPORT.read_text(encoding="utf-8"))

no_results = no_rewrite["results"]
re_results = rewrite["results"]
re_errors = [item for item in re_results if "error" in item]
re_ok = [item for item in re_results if "error" not in item]
common_ids = {item["case_id"] for item in re_ok}
no_common = [item for item in no_results if item.get("case_id") in common_ids]

def avg(rows, metric):
    vals = [item.get("scores", {}).get(metric) for item in rows]
    vals = [value for value in vals if isinstance(value, (int, float))]
    return round(statistics.mean(vals), 3) if vals else None

metrics = ["overall", "recall_at_3", "recall_at_5", "mrr", "citation_accuracy", "groundedness", "latency_ms"]
print("No rewrite full cases:", len(no_results))
print("Rewrite successful cases:", len(re_ok))
print("Rewrite failed cases:", len(re_errors))
if re_errors:
    print("First rewrite error:", re_errors[0].get("error"))

print("\nFull no-rewrite summary:")
print(json.dumps(no_rewrite["summary"], ensure_ascii=False, indent=2))

print("\nCommon successful subset comparison:")
for metric in metrics:
    if metric == "latency_ms":
        before = round(statistics.mean([x["metadata"]["latency_ms"] for x in no_common]), 3)
        after = round(statistics.mean([x["metadata"]["latency_ms"] for x in re_ok]), 3)
    else:
        before = avg(no_common, metric)
        after = avg(re_ok, metric)
    delta = round(after - before, 3) if before is not None and after is not None else None
    print({"metric": metric, "no_rewrite": before, "with_rewrite": after, "delta": delta})


In [ ]:
print("Case-level before/after on successful rewrite subset")
no_by_id = {item["case_id"]: item for item in no_results}
for item in re_ok:
    cid = item["case_id"]
    before = no_by_id[cid]
    print("=" * 100)
    print(cid, "|", item.get("question"))
    print("no_rewrite:", {
        "overall": before["scores"].get("overall"),
        "recall@5": before["scores"].get("recall_at_5"),
        "mrr": before["scores"].get("mrr"),
        "citation": before["scores"].get("citation_accuracy"),
        "latency_ms": before["metadata"].get("latency_ms"),
    })
    print("with_rewrite:", {
        "overall": item["scores"].get("overall"),
        "recall@5": item["scores"].get("recall_at_5"),
        "mrr": item["scores"].get("mrr"),
        "citation": item["scores"].get("citation_accuracy"),
        "latency_ms": item["metadata"].get("latency_ms"),
    })
    print("rewrite_queries:", item.get("retrieval_queries"))


## 9. Quality Improvement Options and Trade-Offs

| C?i thi?n | L? do ph? h?p v?i d? ?n | Trade-off | ?u ti?n |
|---|---|---|---|
| **B?t query rewrite c? ki?m so?t** | C?u h?i ??i th??ng nh? ???nh ng??i b? t?i g?? c?n ???c chuy?n th?nh thu?t ng? ph?p l? v? t?n lu?t/?i?u li?n quan. Smoke test cho th?y recall/MRR c? xu h??ng t?ng tr?n subset ch?y th?nh c?ng. | T?ng latency, t?ng s? l?n g?i LLM, c? th? h?t quota/API cost; rewrite sai c? th? l?m l?ch ? ng??i d?ng. | Cao, nh?ng c?n cache v? gi?i h?n s? query. |
| **Rule-based legal alias expansion** | Nhi?u c?u h?i c? alias: ???nh ng??i? ? ?c? ? g?y th??ng t?ch?, ?ngh? ngang? ? ???n ph??ng ch?m d?t H?L? tr?i ph?p lu?t?. Kh?ng c?n API. | Ph?i b?o tr? dictionary; d? thi?u synonym; rule qu? m?nh c? th? bias sai l?nh v?c. | R?t cao v? mi?n ph? v? gi?m ph? thu?c LLM. |
| **Metadata filtering / document pinning** | Khi query nh?c ho?c rewrite ra ?B? lu?t H?nh s??, ?B? lu?t Lao ??ng?, n?n ?u ti?n chunk thu?c ??ng v?n b?n thay v? ?? vector/BM25 l?n sang ngh? ??nh/th?ng t?. | N?u filter qu? ch?t c? th? b? s?t ngu?n ??ng khi query m? h?; c?n metadata s?ch. | Cao. |
| **Parent-child retrieval** | Search b?ng chunk nh? nh?ng ??a c? ?i?u ??y ??/parent context v?o LLM. Ph? h?p v?n b?n ph?p lu?t v? ?i?u/Kho?n c? ?i?u ki?n v? ngo?i l?. | Context d?i h?n, t?n token h?n; c?n ch?ng duplicate. | Cao. |
| **Rerank theo t?n lu?t + s? ?i?u** | Hi?n nhi?u case fail v? top source c?ng ch? ?? nh?ng sai v?n b?n/?i?u. Feature match `document_title`, `article_number`, `doc_number` s? gi?p citation. | Rule-based rerank d? overfit; c?n eval sau m?i thay ??i. | Cao. |
| **Local cross-encoder reranker** | C? th? c?i thi?n th? t? top candidate m? kh?ng t?n API n?u ch?y local. | Ch?m h?n, t?n RAM/CPU/GPU; c?n t?i model; kh?ng n?n rerank qu? nhi?u candidate. | Trung b?nh. |
| **Improve Vietnamese normalization/tokenization** | BM25 hi?n y?u v?i c?u h?i ??i th??ng. Chu?n h?a d?u, vi?t t?t, c?m ph?p l? gi?p lexical search t?t h?n. | Tokenizer ti?ng Vi?t kh?ng ph?i l?c n?o c?ng t?t v?i s? hi?u/?i?u lu?t; th?m dependency. | Trung b?nh-cao. |
| **Hard negative evaluation loop** | C?c case th?p ?i?m trong notebook l? hard negatives th?t. D?ng ch?ng ?? ch?nh query expansion/rerank r?i ?o l?i. | T?n c?ng g?n nh?n v? d? overfit v?o 56 case n?u kh?ng m? r?ng dataset. | Cao. |
| **Answer guard d?a tr?n retrieved source** | N?u expected/citation source kh?ng v?o top-k, model n?n n?i thi?u ngu?n ho?c gi?m ?? ch?c ch?n. | C? th? l?m c?u tr? l?i th?n tr?ng h?n, ?t ?m??t? h?n. | Cao cho Legal RAG. |
| **Cache query rewrite/retrieval** | Nh?ng c?u h?i ph? bi?n l?p l?i, cache gi?p gi?m latency/quota. | C?n invalidation theo corpus version, prompt version, model v? settings; cache sai c? th? tr? ngu?n c?. | Trung b?nh. |

T?m l?i: h??ng n?n l?m tr??c l? **legal alias expansion + metadata/document pinning + rerank theo lu?t/?i?u**, v? c?c c?ch n?y g?n nh? kh?ng t?ng chi ph? ch?y. Query rewrite c? ?ch nh?ng ph?i d?ng c? ki?m so?t v? trade-off r?t r?: **ch?t l??ng t?ng ??i l?y latency, API cost v? quota risk**.